In [0]:
from pyspark.sql.functions import *
from src.logic.transformation_functions import *

In [0]:
# reading silver data
order_silver = spark.read.table("ecommerce_project.silver_ecom.orders_silver")

order_items_silver = spark.read.table("ecommerce_project.silver_ecom.order_items_silver")

customers_silver = spark.read.table("ecommerce_project.silver_ecom.customers_silver")

order_enriched_silver = spark.read.table("ecommerce_project.silver_ecom.orders_enriched_silver")

## Gold Layer

**sales_summary_gold:** 
- Total orders, 
- Total revenue, 
- Average order value, 
- Daily revenue trend

**customer_performance_gold:** 
- Total spend per customer, 
- Order count per customer 

**product_performance_gold:** 
- Revenue per product, 
- Quantity sold per product 

### Sales Summary 

In [0]:
sales_summary_gold = (
    order_silver
    .join(order_enriched_silver, on="order_id", how="inner")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("order_level_total_amt"), 2).alias("total_revenue"),
        round(avg("order_level_total_amt"), 2).alias("average_order_value")
    )
)

sales_summary_gold.display()


In [0]:
save_df_to_delta(sales_summary_gold, "overwrite", "ecommerce_project.gold_ecom.sales_summary_gold")

### Daily Revenue Trends

In [0]:
daily_revenue_trend_gold = (
    order_silver
    .join(order_enriched_silver, on="order_id", how="inner")
    .groupBy("order_date")
    .agg(
        round(sum("order_level_total_amt"),2).alias("daily_revenue")
    )
    .orderBy("order_date")
)

daily_revenue_trend_gold.display()

In [0]:
# saving daily_revenue_trend_gold table
save_df_to_delta(daily_revenue_trend_gold, "overwrite", "ecommerce_project.gold_ecom.daily_revenue_trend_gold")

### Customer Performance

In [0]:
customer_performance_gold = (
    order_silver
    .join(order_enriched_silver, on="order_id", how="inner")
    .groupBy("customer_id")
    .agg(
        round(sum("order_level_total_amt"), 2).alias("total_spend"),
        countDistinct("order_id").alias("order_count")
    )
)

customer_performance_gold.display()

In [0]:
# saving customer performance table
save_df_to_delta(customer_performance_gold, "overwrite", "ecommerce_project.gold_ecom.customer_performance_gold")

### Product Performance

In [0]:
product_performance_gold = (
    order_items_silver
    .groupBy("product_name")
    .agg(
        round(sum("total_price"), 2).alias("product_revenue"),
        sum("quantity").alias("quantity_sold")
    )
)

product_performance_gold.display()

In [0]:
# saving product performance table
save_df_to_delta(product_performance_gold, "overwrite", "ecommerce_project.gold_ecom.product_performance_gold")